# 01 — Validation: can the raw data reproduce DB's official punctuality?

Before trusting any finding, check the data against the official figure.

**Steps**
1. **Coverage** — are any days or months missing data?
2. **Reconciliation** — our monthly long-distance punctuality next to DB's published figure.
3. **Gap tests** — which definition choices close the gap?
4. **Cancellations** — does cancelling late trains flatter the statistic?

**Data:** `data/db_punctuality.duckdb`, built by `src/build_db.py`.
**Definitions:** `docs/kpi_definitions.md`. Punctual = arrived less than 6 minutes late (delay of 5:59 or less).

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIG = ROOT / "notebooks" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(str(ROOT / "data" / "db_punctuality.duckdb"), read_only=True)

def q(sql):
    return con.execute(sql).df()

plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False})
q("select min(planned_arrival) first_stop, max(planned_arrival) last_stop, count(*) stops from stg_stops")

## 1. Coverage

The dataset is collected from DB's live timetable API (application programming interface) by a third party. Gaps in collection show up as days with far fewer stops than usual. Those days must be flagged before comparing anything.

In [ ]:
daily = q('''
    select cast(planned_arrival as date) as day, segment, count(*) as stops
    from stg_stops group by all order by day''')
pivot = daily.pivot(index="day", columns="segment", values="stops")

median = pivot.median()
thin = pivot[(pivot < 0.5 * median).any(axis=1)]
print(f"Days with under half the usual number of stops: {len(thin)}")
display(thin)

ax = pivot.plot(subplots=True, figsize=(10, 5), legend=True, title="Planned stops per day")
plt.tight_layout(); plt.savefig(FIG / "01_coverage_daily.png", dpi=150); plt.show()

In [ ]:
monthly_cov = q('''
    select month,
           count(distinct station_id) as stations,
           count(distinct cast(planned_arrival as date)) as days_with_data,
           count(distinct journey_id) filter (where segment = 'long_distance') as long_distance_journeys
    from stg_stops group by month order by month''')
monthly_cov

In [ ]:
# Months to exclude from the comparison: fewer than 25 days of data
EXCLUDE = monthly_cov.loc[monthly_cov.days_with_data < 25, "month"].astype(str).tolist()
print("Excluded months:", EXCLUDE or "none")
FULL_MONTHS = ", ".join(f"DATE '{m}'" for m in monthly_cov.loc[monthly_cov.days_with_data >= 25, "month"].astype(str))

## 2. Reconciliation with DB's published figure

DB publishes a monthly "betriebliche Pünktlichkeit" (operational punctuality) for long-distance trains (ICE, IC, EC). Figures below were copied from DB's punctuality page on 30 Oct 2026. Recent months are marked provisional by DB.

In [ ]:
official = pd.DataFrame({
    "month": pd.to_datetime(["2025-10-01","2025-11-01","2025-12-01","2026-01-01","2026-02-01","2026-03-01",
                             "2026-04-01","2026-05-01","2026-06-01","2026-07-01","2026-08-01","2026-09-01"]),
    "db_official_pct": [51.5, 54.5, 61.7, 52.1, 59.4, 62.1, 64.4, 61.1, 52.6, 52.6, 54.2, 52.4],
})

ours = q('''select month, official_punctuality_pct as ours_pct, honest_punctuality_pct, cancelled_pct
            from mart_monthly_kpis where segment = 'long_distance' order by month''')
ours["month"] = pd.to_datetime(ours["month"])

recon = official.merge(ours, on="month", how="left")
recon["gap_pts"] = (recon.ours_pct - recon.db_official_pct).round(1)
recon["excluded"] = recon.month.dt.strftime("%Y-%m-%d").isin(EXCLUDE)
recon

In [ ]:
r = recon[~recon.excluded]
print(f"Mean gap: {r.gap_pts.mean():.1f} points | spread (standard deviation): {r.gap_pts.std():.1f}")
print(f"Correlation of monthly ups and downs: {r.ours_pct.corr(r.db_official_pct):.2f}")

fig, ax = plt.subplots()
ax.plot(r.month, r.db_official_pct, marker="o", label="DB official")
ax.plot(r.month, r.ours_pct, marker="o", label="Our raw data")
ax.set_ylabel("% of stops less than 6 min late"); ax.set_title("Long-distance punctuality: ours vs DB")
ax.legend(); plt.tight_layout(); plt.savefig(FIG / "02_reconciliation.png", dpi=150); plt.show()

**How to read it:** a steady gap with a high correlation means our data tracks DB's trend but measures something slightly different. A gap that jumps around means a data problem.

## 3. Gap tests

Each variant changes one definition choice. The one that brings the mean gap closest to zero shows which choice matters most.

In [ ]:
variants = {
    "A. All stops, ICE + IC + EC (baseline)":
        "segment = 'long_distance'",
    "B. ICE only":
        "train_type = 'ICE'",
    "C. ICE + IC, no EC (EC includes foreign operators)":
        "train_type in ('ICE', 'IC')",
    "D. Planned stops only (no added stops)":
        "segment = 'long_distance' and not coalesce(is_additional_stop, false)",
    "E. Stations present in every month":
        '''segment = 'long_distance' and station_id in (
              select station_id from stg_stops where month in ({FULL_MONTHS}) group by station_id
              having count(distinct month) = (select count(distinct month) from stg_stops where month in ({FULL_MONTHS})))''',
    "F. Final stop of each journey only":
        '''segment = 'long_distance' and (journey_id, stop_seq) in (
              select journey_id, max(stop_seq) from stg_stops
              where segment = 'long_distance' group by journey_id)''',
}

rows = []
for name, where in variants.items():
    where = where.replace("{FULL_MONTHS}", FULL_MONTHS)
    df = q(f'''select month, round(100 * avg(case when not is_cancelled then (delay_in_min < 6)::int end), 1) as pct
               from stg_stops where {where} group by month''')
    df["month"] = pd.to_datetime(df["month"])
    m = official.merge(df, on="month")
    m = m[~m.month.dt.strftime("%Y-%m-%d").isin(EXCLUDE)]
    rows.append({"variant": name,
                 "mean_gap_pts": round((m.pct - m.db_official_pct).mean(), 1),
                 "mean_abs_gap_pts": round((m.pct - m.db_official_pct).abs().mean(), 1)})

gap_tests = pd.DataFrame(rows).sort_values("mean_abs_gap_pts")
gap_tests

**Write down the conclusion here**, for example: "Variant X closes the gap to Y points, so the difference comes mainly from Z." Whatever is left over is likely to be DB's own weighting, which isn't published.

## 4. Cancellations: does cancelling late trains flatter the statistic?

DB's figure leaves cancelled stops out. In September 2025 Der Spiegel reported internal staff chats saying an ICE was ended early in Cologne to improve the statistics. DB rejected the claim and said cutting a very late train short can be operationally sensible.

This section asks what the data shows:
- How are journeys split between running fully, being cut short, skipping stops and being fully cancelled?
- Does the chance of the rest of a journey being cancelled rise with how late the train is?
- How much higher is the official figure than one that counts cancelled stops?

**Caution:** a link between delay and cancellation does not show intent. Late trains are also cut short to stop delays spreading.

In [ ]:
journey_kinds = q('''
    with j as (
        select journey_id, month,
               min(case when is_cancelled then stop_seq end) as first_cancel,
               max(case when not is_cancelled then stop_seq end) as last_served
        from stg_stops where segment = 'long_distance' group by journey_id, month)
    select month,
           case when first_cancel is null then 'ran fully'
                when last_served is null then 'fully cancelled'
                when first_cancel > last_served then 'cut short'
                else 'skipped stops mid-route' end as kind,
           count(*) as journeys
    from j group by all order by month''')
kinds = journey_kinds.pivot(index="month", columns="kind", values="journeys")
kinds_pct = (kinds.div(kinds.sum(axis=1), axis=0) * 100).round(1)
kinds_pct

In [ ]:
cut_by_delay = q('''
    with s as (
        select journey_id, stop_seq, is_cancelled, delay_in_min,
               lead(is_cancelled) over w as next_cancelled,
               bool_and(is_cancelled) over (partition by journey_id order by stop_seq
                    rows between 1 following and unbounded following) as rest_cancelled,
               max(stop_seq) over (partition by journey_id) as last_seq
        from stg_stops where segment = 'long_distance'
        window w as (partition by journey_id order by stop_seq))
    select case when delay_in_min < 6  then '1: 0-5 min'
                when delay_in_min < 16 then '2: 6-15 min'
                when delay_in_min < 31 then '3: 16-30 min'
                when delay_in_min < 61 then '4: 31-60 min'
                else '5: over 60 min' end as delay_at_stop,
           count(*) as stops,
           round(100 * avg((next_cancelled and rest_cancelled)::int), 2) as pct_rest_cancelled
    from s
    where not is_cancelled and stop_seq < last_seq
    group by 1 order by 1''')
base = cut_by_delay.pct_rest_cancelled.iloc[0]
cut_by_delay["times_more_likely_than_on_time"] = (cut_by_delay.pct_rest_cancelled / base).round(1)
cut_by_delay

In [ ]:
fig, ax = plt.subplots()
ax.bar(cut_by_delay.delay_at_stop.str[3:], cut_by_delay.pct_rest_cancelled)
ax.set_xlabel("Delay when the train reached the stop"); ax.set_ylabel("% where the rest of the journey was cancelled")
ax.set_title("Late long-distance trains are far more often cut short")
plt.tight_layout(); plt.savefig(FIG / "03_cut_short_by_delay.png", dpi=150); plt.show()

In [ ]:
# What the cut-short journeys were doing to the statistic:
# their cancelled stops are dropped from the official figure. Count them as late instead.
effect = q('''
    with j as (
        select journey_id,
               min(case when is_cancelled then stop_seq end) as first_cancel,
               max(case when not is_cancelled then stop_seq end) as last_served
        from stg_stops where segment = 'long_distance' group by journey_id),
    cut as (select journey_id from j where first_cancel > last_served)
    select s.month,
           round(100 * avg(case when not s.is_cancelled then (s.delay_in_min < 6)::int end), 1) as official_pct,
           round(100 * avg(case when s.is_cancelled and s.journey_id in (select journey_id from cut) then 0
                                when s.is_cancelled then null
                                else (s.delay_in_min < 6)::int end), 1) as cut_short_stops_counted_late_pct,
           round(100 * avg(case when s.is_cancelled then 0 else (s.delay_in_min < 6)::int end), 1) as all_cancellations_counted_late_pct
    from stg_stops s where s.segment = 'long_distance'
    group by s.month order by s.month''')
effect["boost_from_cut_short_pts"] = (effect.official_pct - effect.cut_short_stops_counted_late_pct).round(1)
effect

## Findings (fill in after running)

- **Coverage:** …
- **Reconciliation:** our figure runs about … points below DB's, and tracks its monthly movement (correlation …).
- **Gap:** the variant that closes most of the gap is …
- **Cancellations:** a train over 60 minutes late is about … times more likely to have the rest of its journey cancelled than one on time. Cut-short journeys raise the official figure by about … points.